# Explore Banking77

Understand the messages, 77 intent labels, class balance, and sequence lengths before choosing a model. The first run downloads the dataset through Hugging Face.

In [ ]:
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
from datasets import load_dataset

dataset = load_dataset('PolyAI/banking77')
dataset

## Dataset and label names

Banking77 provides official train and test splits. The integer `label` column uses the names stored in its `ClassLabel` feature.

In [ ]:
class_names = dataset['train'].features['label'].names
print('Train examples:', len(dataset['train']))
print('Test examples:', len(dataset['test']))
print('Intent labels:', len(class_names))
for index, name in enumerate(class_names):
    print(f'{index:2d}: {name}')

## Read actual examples

Look for ambiguous wording, shared vocabulary, spelling variation, and context that distinguishes neighboring intents.

In [ ]:
for example in dataset['train'].select(range(12)):
    print(f"[{class_names[example['label']]}] {example['text']}")

## Class balance

Macro F1 gives every label equal importance, even when class counts differ.

In [ ]:
counts = Counter(dataset['train']['label'])
ordered_counts = np.array([counts[index] for index in range(len(class_names))])
print('Minimum / median / maximum examples:', ordered_counts.min(), np.median(ordered_counts), ordered_counts.max())

plt.figure(figsize=(16, 5))
plt.bar(class_names, ordered_counts)
plt.xticks(rotation=90, fontsize=6)
plt.ylabel('Training examples')
plt.title('Banking77 training class counts')
plt.tight_layout();

## Text lengths

The project defaults to 64 tokens. A simple whitespace count is not identical to either project tokenizer, but it gives a useful first estimate.

In [ ]:
lengths = np.array([len(text.split()) for text in dataset['train']['text']])
for percentile in (50, 90, 95, 99, 100):
    print(f'{percentile:3d}th percentile: {np.percentile(lengths, percentile):.0f} whitespace tokens')

plt.hist(lengths, bins=30)
plt.xlabel('Whitespace-token count')
plt.ylabel('Messages')
plt.title('Training message lengths');

## Compare easily confused intents

Change the names below and inspect examples. This is often the fastest way to understand what a high-quality classifier must learn.

In [ ]:
selected_intents = [
    'declined_cash_withdrawal',
    'cash_withdrawal_not_recognised',
    'wrong_exchange_rate_for_cash_withdrawal',
]
for intent in selected_intents:
    if intent not in class_names:
        print(f'Label not present in this dataset version: {intent}')
        continue
    label_id = class_names.index(intent)
    examples = [text for text, label in zip(dataset['train']['text'], dataset['train']['label']) if label == label_id][:5]
    print(f'\n{intent}')
    for text in examples:
        print(' -', text)

## Questions to answer before training

1. Which label pairs appear most semantically similar?
2. Is accuracy alone enough when support differs by class?
3. Would a word tokenizer handle spelling variation and rare words as well as DistilBERT's subword tokenizer?
4. What should happen when a message is unrelated to all 77 labels?
5. Why must the official test set remain separate from model and threshold selection?